# Pratica 05 — Classificazione di testi

**Corso:** NLP Classico con Python · **Lezione 05** · **Tempo stimato:** 75-90 minuti
**Notebook di lezione:** `notebooks_clase/clase05_clasificacion_de_textos_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- L'esercizio 1 usa un corpus minuscolo definito nella cella di dati; gli altri usano i ticket di `datos/` (in Colab, carica la cartella `datos/`). Sono testi sintetici scritti per il corso (*esempio illustrativo*), con spagnolo e italiano quasi paralleli: non sono due esperimenti indipendenti.
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: `numpy`, `pandas` e `scikit-learn`.

---
## Cella 0 — Setup e dati

In [ ]:
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

ARCHIVO_TICKETS = {"spa": "tickets_es.csv", "ita": "tickets_it.csv"}
tickets = {idioma: pd.read_csv(cargar_datos(archivo)) for idioma, archivo in ARCHIVO_TICKETS.items()}
ETIQUETAS = sorted(tickets["spa"]["categoria"].unique())


def separar_veredicto(texto):
    """Separa l'ultima frase se ha 4 parole o meno (verdetto o etichetta di oggetto)."""
    m = re.search(r"(?<=[.?!])\s+([^.?!]+[.?!])\s*$", texto)
    if m and len(m.group(1).split()) <= 4:
        return texto[:m.start()].strip(), m.group(1).strip()
    return texto, ""


print({idioma: df.shape for idioma, df in tickets.items()})
print("Librerie caricate correttamente.")

In [ ]:
# datos-es
NB_P = {
    "spa": {"cuenta": ["olvidé mi contraseña", "no puedo entrar en mi cuenta"],
            "envio": ["mi paquete no llega", "el paquete llega tarde"]},
    "ita": {"cuenta": ["ho dimenticato la password", "non riesco ad entrare nel mio account"],
            "envio": ["il mio pacco non arriva", "il pacco arriva in ritardo"]},
}
TEXTO_NB = {"spa": "no llega mi cuenta", "ita": "non arriva il mio account"}
CUOTAS_P = {"facturacion": 80, "envio": 40, "cuenta": 15, "devoluciones": 8}

## Esercizio 1 — Naive Bayes con lisciamento di Laplace, a mano

Con il corpus minuscolo `NB_P` (due ticket per classe, `cuenta` e `envio`) e il ticket nuovo di `TEXTO_NB`:

- completa `prob_palabra(recuento, n_clase, V, alpha)`: `(conteggio + alpha) / (n_classe + alpha · V)`,
- completa il **priori** di `puntuacion` (frazione di documenti della classe).

`puntuacion` moltiplica il priori per la probabilità di ogni parola del ticket. Si verifica con `MultinomialNB(alpha=1)` nelle due lingue. Prima di programmare, calcola a mano `P(non | cuenta)` in italiano: la classe `cuenta` ha 11 parole e il vocabolario 16.

In [ ]:
def prob_palabra(recuento, n_clase, V, alpha=1.0):
    return None      # TODO: (conteggio + alpha) / (n_classe + alpha · V)


def preparar_nb(idioma):
    docs = {clase: [re.findall(r"\w+", d.lower()) for d in ds] for clase, ds in NB_P[idioma].items()}
    vocab = sorted({t for ds in docs.values() for d in ds for t in d})
    return docs, vocab


def puntuacion(clase, texto, docs, V, alpha=1.0):
    tokens_clase = [t for d in docs[clase] for t in d]
    priori = None    # TODO: documenti della classe / documenti totali
    p = priori
    for palabra in re.findall(r"\w+", texto.lower()):
        p = p * prob_palabra(tokens_clase.count(palabra), len(tokens_clase), V, alpha)
    return p

# Controllo automatico
assert prob_palabra(2, 6, 11) is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(prob_palabra(2, 6, 11), 3 / 17), "(2 + 1) / (6 + 11) = 3/17"
assert np.isclose(prob_palabra(0, 6, 11), 1 / 17) and prob_palabra(0, 6, 11, alpha=0) == 0, "con alpha = 0 non c'è lisciamento e la probabilità è 0"
for idioma in ("spa", "ita"):
    docs, vocab = preparar_nb(idioma)
    V = len(vocab)
    texto = TEXTO_NB[idioma]
    assert puntuacion("cuenta", texto, docs, V) is not None and puntuacion("envio", texto, docs, V) is not None, "Completa le righe contrassegnate con TODO"
    puntos = {c: puntuacion(c, texto, docs, V) for c in sorted(docs)}
    total = sum(puntos.values())
    posteriores = np.array([puntos[c] / total for c in sorted(docs)])
    textos = [d for ds in NB_P[idioma].values() for d in ds]
    etiquetas = [c for c, ds in NB_P[idioma].items() for _ in ds]
    cv = CountVectorizer(token_pattern=r"\w+")
    nb = MultinomialNB(alpha=1.0).fit(cv.fit_transform(textos), etiquetas)
    assert list(nb.classes_) == sorted(docs)
    assert np.allclose(posteriores, nb.predict_proba(cv.transform([texto]))[0]), "i posteriori devono coincidere con MultinomialNB(alpha=1): rivedi il priori e la formula"
    assert np.isclose(puntos["cuenta"] / total + puntos["envio"] / total, 1.0)
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Precisione, richiamo e F1 dalla matrice di confusione

`M` è una matrice di confusione con **classi sbilanciate** (righe = classe reale, colonne = classe predetta, nell'ordine di `ETIQUETAS`). Calcola, senza usare `scikit-learn`:

- `precision`, `recall` e `f1`: array con un valore per classe,
- `f1_macro`: media semplice degli F1,
- `f1_pond`: media degli F1 pesata per il supporto (numero di esempi reali di ogni classe).

Prima di programmare, calcola a mano il richiamo di `devoluciones` (riga 2).

In [ ]:
M = np.array([[24, 3, 2, 1],      # cuenta
              [1, 6, 1, 2],       # devoluciones
              [2, 0, 17, 1],      # envio
              [1, 2, 0, 37]])     # facturacion
print(ETIQUETAS)

precision = None    # TODO: diagonale / somma di ogni colonna
recall = None       # TODO: diagonale / somma di ogni riga
f1 = None           # TODO: 2 · P · R / (P + R)
f1_macro = None     # TODO
f1_pond = None      # TODO: ponderato per il supporto (somma di ogni riga)

# Controllo automatico
assert all(v is not None for v in (precision, recall, f1, f1_macro, f1_pond)), "Completa le righe contrassegnate con TODO"
assert np.isclose(recall[1], 6 / 10), "il richiamo di devoluciones è 6/10 = 0,6"
y_real = [ETIQUETAS[i] for i in range(4) for j in range(4) for _ in range(M[i, j])]
y_pred = [ETIQUETAS[j] for i in range(4) for j in range(4) for _ in range(M[i, j])]
p, r, f, s = precision_recall_fscore_support(y_real, y_pred, labels=ETIQUETAS)
assert np.allclose(precision, p), "precisione = diagonale / somma della colonna"
assert np.allclose(recall, r), "richiamo = diagonale / somma della riga"
assert np.allclose(f1, f), "F1 = 2 · P · R / (P + R)"
assert np.isclose(f1_macro, f1_score(y_real, y_pred, average="macro")), "l'F1 macro è la media semplice degli F1"
assert np.isclose(f1_pond, f1_score(y_real, y_pred, average="weighted")), "il ponderato usa come pesi il supporto di ogni classe"
assert f1_macro < f1_pond, "qui il ponderato è maggiore: la classe piccola (devoluciones) è la peggiore"
print("Esercizio 2 corretto ✔")

## Esercizio 3 — `Pipeline` e validazione incrociata stratificata

Completa:

- `crear_pipeline_svm()`: una `Pipeline` con due passi, `"tfidf"` (`TfidfVectorizer(ngram_range=(1, 2))`) e `"clf"` (`LinearSVC()`),
- `validar(df)`: restituisce le 5 accuratezze di `cross_val_score` con `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`, passando **l'intera `Pipeline`** (così il vettorizzatore si adatta solo con l'addestramento di ogni partizione).

Il controllo ripete il calcolo a mano, adattando in ogni partizione solo con l'addestramento, e deve dare lo stesso risultato.

In [ ]:
def crear_pipeline_svm():
    return None      # TODO: Pipeline([("tfidf", ...), ("clf", ...)])


def validar(df):
    return None      # TODO: cross_val_score(..., cv=StratifiedKFold(...))

# Controllo automatico
pipe = crear_pipeline_svm()
assert pipe is not None, "Completa le righe contrassegnate con TODO"
assert isinstance(pipe, Pipeline) and [n for n, _ in pipe.steps] == ["tfidf", "clf"], "la Pipeline deve avere i passi «tfidf» e «clf»"
assert isinstance(pipe.named_steps["tfidf"], TfidfVectorizer) and pipe.named_steps["tfidf"].ngram_range == (1, 2)
assert isinstance(pipe.named_steps["clf"], LinearSVC)
assert not hasattr(pipe.named_steps["tfidf"], "vocabulary_"), "la Pipeline si restituisce senza adattarla"
for idioma in ("spa", "ita"):
    df = tickets[idioma]
    puntos = validar(df)
    assert puntos is not None, "Completa le righe contrassegnate con TODO"
    assert len(puntos) == 5, "5 partizioni, 5 valori"
    manual = []
    for idx_ent, idx_prueba in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(df["texto"], df["categoria"]):
        modelo = crear_pipeline_svm().fit(df["texto"].iloc[idx_ent], df["categoria"].iloc[idx_ent])
        manual.append(modelo.score(df["texto"].iloc[idx_prueba], df["categoria"].iloc[idx_prueba]))
    assert np.allclose(puntos, manual), "deve coincidere con il calcolo manuale: passa l'intera Pipeline a cross_val_score"
    assert puntos.mean() > 0.8, "con TF-IDF e LinearSVC ci si aspetta più di 0,8"
    print(idioma, "accuratezza media", round(puntos.mean(), 3), "±", round(puntos.std(), 3))
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Creare uno sbilanciamento e misurare ciò che l'accuratezza nasconde

Completa:

- `submuestrear(df, cuotas, semilla)`: tiene `cuotas[categoria]` ticket di ogni categoria (usa `groupby` e `sample(n=..., random_state=semilla)`) e restituisce un `DataFrame` con l'indice reimpostato,
- `evaluar(des)`: addestra una regressione logistica in una `Pipeline` (TF-IDF con unigrammi e bigrammi) **senza** pesi e un'altra con `class_weight="balanced"`, e restituisce `(accuratezza_senza, f1_macro_senza, f1_macro_balanced)` sul test.

Prima di programmare: che accuratezza avrebbe un modello che dice sempre `facturacion` con `CUOTAS_P`? (80 ticket su 143.)

In [ ]:
def submuestrear(df, cuotas, semilla=42):
    return None      # TODO: concatenare g.sample(n=cuotas[c], ...) di ogni categoria


def evaluar(des):
    X_ent, X_prueba, y_ent, y_prueba = train_test_split(des["texto"], des["categoria"], test_size=0.3, stratify=des["categoria"], random_state=42)
    exactitud_sin = None        # TODO: accuratezza della regressione logistica senza pesi
    f1_sin = None               # TODO: F1 macro senza pesi
    f1_balanced = None          # TODO: F1 macro con class_weight="balanced"
    return exactitud_sin, f1_sin, f1_balanced

# Controllo automatico
for idioma in ("spa", "ita"):
    des = submuestrear(tickets[idioma], CUOTAS_P)
    assert des is not None, "Completa le righe contrassegnate con TODO"
    assert des["categoria"].value_counts().to_dict() == CUOTAS_P, "il DataFrame deve avere esattamente le quote richieste"
    assert list(des.index) == list(range(143)), "reimposta l'indice (reset_index(drop=True))"
    assert des.equals(submuestrear(tickets[idioma], CUOTAS_P)), "con lo stesso seme il risultato deve ripetersi"
    res = evaluar(des)
    assert all(v is not None for v in res), "Completa le righe contrassegnate con TODO"
    exactitud_sin, f1_sin, f1_bal = res
    assert exactitud_sin > f1_sin + 0.15, "senza pesi l'accuratezza dovrebbe essere molto maggiore dell'F1 macro"
    assert f1_bal > f1_sin + 0.1, "class_weight=\"balanced\" dovrebbe migliorare l'F1 macro"
    print(idioma, "accuratezza senza pesi", round(exactitud_sin, 3), "| F1 macro senza pesi", round(f1_sin, 3), "| F1 macro balanced", round(f1_bal, 3))
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Leggere i coefficienti e scoprire gli artefatti

Completa:

- `top_palabras(pipe, k)`: data una `Pipeline` già addestrata (TF-IDF + `LogisticRegression`), restituisce un dizionario `classe → lista delle k parole con il coefficiente maggiore` (dalla maggiore alla minore),
- `n_veredictos(df)`: numero di ticket il cui testo ha una frase finale corta secondo `separar_veredicto` (il secondo elemento restituito non è vuoto).

È la lettura critica della lezione: le parole dell'elenco si interpretano, e i verdetti e le etichette di oggetto sono un artefatto dei dati.

In [ ]:
def top_palabras(pipe, k=8):
    palabras = pipe.named_steps["tfidf"].get_feature_names_out()
    coef = pipe.named_steps["clf"].coef_
    clases = pipe.named_steps["clf"].classes_
    return None      # TODO: {classe: [k parole con il coefficiente maggiore]}


def n_veredictos(df):
    return None      # TODO: conta i ticket con verdetto

# Controllo automatico
FACTURACION = {"spa": {"cobro", "factura", "pago", "importe", "tarjeta", "cargo", "cobrado", "pagos"},
               "ita": {"addebito", "fattura", "pagamento", "importo", "addebitato", "carta", "pagato", "pagare"}}
for idioma in ("spa", "ita"):
    df = tickets[idioma]
    pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000))]).fit(df["texto"], df["categoria"])
    top = top_palabras(pipe, k=8)
    assert top is not None and n_veredictos(df) is not None, "Completa le righe contrassegnate con TODO"
    assert sorted(top) == ETIQUETAS and all(len(v) == 8 for v in top.values()), "una lista di 8 parole per classe"
    fila = list(pipe.named_steps["clf"].classes_).index("facturacion")
    coef = pipe.named_steps["clf"].coef_[fila]
    palabras = list(pipe.named_steps["tfidf"].get_feature_names_out())
    assert top["facturacion"][0] == palabras[int(np.argmax(coef))], "la prima parola deve essere quella con il coefficiente maggiore"
    coefs_top = [coef[palabras.index(p)] for p in top["facturacion"]]
    assert coefs_top == sorted(coefs_top, reverse=True), "ordina dal coefficiente maggiore al minore"
    assert len(set(top["facturacion"][:5]) & FACTURACION[idioma]) >= 3, "le parole di fatturazione dovrebbero dominare l'elenco"
    assert n_veredictos(df) == sum(1 for t in df["texto"] if separar_veredicto(t)[1]) and n_veredictos(df) > 100, "conta i ticket con frase finale corta"
    print(idioma, "facturacion:", top["facturacion"], "| ticket con verdetto:", n_veredictos(df))
print("Esercizio 5 corretto ✔")

---
## Sfida finale — quanto vale il verdetto?

Integra quanto appreso. Completa `diferencia_veredicto(df)`, che deve restituire `(con, sin)`:

- `cuerpos`: la serie dei testi **senza** la frase finale corta (usa `separar_veredicto`),
- `con`: accuratezza media della validazione incrociata stratificata (5 partizioni, `shuffle=True`, `random_state=42`) con una `Pipeline` TF-IDF + regressione logistica sui testi completi,
- `sin`: la stessa misura su `cuerpos`.

Verifica che in entrambe le lingue togliere il verdetto **abbassi** l'accuratezza. Poi scrivi nella cella delle conclusioni cosa significa questo per interpretare i coefficienti.

In [ ]:
def diferencia_veredicto(df):
    cuerpos = None      # TODO: testi senza la frase finale corta
    con = None          # TODO: accuratezza media (CV stratificata) con i testi completi
    sin = None          # TODO: accuratezza media (CV stratificata) con «cuerpos»
    return con, sin

def _cv(textos, y):
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", LogisticRegression(max_iter=1000))])
    return cross_val_score(pipe, textos, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)).mean()


for idioma in ("spa", "ita"):
    df = tickets[idioma]
    resultado = diferencia_veredicto(df)
    assert resultado is not None, "Completa le righe contrassegnate con TODO"
    con, sin = resultado
    assert con is not None and sin is not None, "Completa le righe contrassegnate con TODO"
    esperado_sin = _cv(df["texto"].map(lambda t: separar_veredicto(t)[0]), df["categoria"])
    assert np.isclose(con, _cv(df["texto"], df["categoria"])), "valore con verdetto: validazione incrociata sui testi completi"
    assert np.isclose(sin, esperado_sin), "valore senza verdetto: validazione incrociata sui testi senza la frase finale corta"
    assert con - sin > 0.01, "togliere il verdetto dovrebbe abbassare l'accuratezza"
    print(idioma, "con verdetto", round(con, 4), "| senza verdetto", round(sin, 4))
print("Sfida finale corretta ✔")

**Le tue conclusioni (scrivi 3-4 frasi):** cosa significa che l'accuratezza scenda togliendo la frase finale? Se un coefficiente alto di `addebito` o `accesso` viene in parte da quella frase, lo useresti per spiegare cosa fa il modello con ticket reali? Quali altri artefatti cercheresti in un tuo corpus prima di interpretare i coefficienti?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def prob_palabra(recuento, n_clase, V, alpha=1.0):
    return (recuento + alpha) / (n_clase + alpha * V)


def preparar_nb(idioma):
    docs = {clase: [re.findall(r"\w+", d.lower()) for d in ds] for clase, ds in NB_P[idioma].items()}
    vocab = sorted({t for ds in docs.values() for d in ds for t in d})
    return docs, vocab


def puntuacion(clase, texto, docs, V, alpha=1.0):
    tokens_clase = [t for d in docs[clase] for t in d]
    priori = len(docs[clase]) / sum(len(ds) for ds in docs.values())
    p = priori
    for palabra in re.findall(r"\w+", texto.lower()):
        p = p * prob_palabra(tokens_clase.count(palabra), len(tokens_clase), V, alpha)
    return p

# Controllo automatico
assert prob_palabra(2, 6, 11) is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(prob_palabra(2, 6, 11), 3 / 17), "(2 + 1) / (6 + 11) = 3/17"
assert np.isclose(prob_palabra(0, 6, 11), 1 / 17) and prob_palabra(0, 6, 11, alpha=0) == 0, "con alpha = 0 non c'è lisciamento e la probabilità è 0"
for idioma in ("spa", "ita"):
    docs, vocab = preparar_nb(idioma)
    V = len(vocab)
    texto = TEXTO_NB[idioma]
    assert puntuacion("cuenta", texto, docs, V) is not None and puntuacion("envio", texto, docs, V) is not None, "Completa le righe contrassegnate con TODO"
    puntos = {c: puntuacion(c, texto, docs, V) for c in sorted(docs)}
    total = sum(puntos.values())
    posteriores = np.array([puntos[c] / total for c in sorted(docs)])
    textos = [d for ds in NB_P[idioma].values() for d in ds]
    etiquetas = [c for c, ds in NB_P[idioma].items() for _ in ds]
    cv = CountVectorizer(token_pattern=r"\w+")
    nb = MultinomialNB(alpha=1.0).fit(cv.fit_transform(textos), etiquetas)
    assert list(nb.classes_) == sorted(docs)
    assert np.allclose(posteriores, nb.predict_proba(cv.transform([texto]))[0]), "i posteriori devono coincidere con MultinomialNB(alpha=1): rivedi il priori e la formula"
    assert np.isclose(puntos["cuenta"] / total + puntos["envio"] / total, 1.0)
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
M = np.array([[24, 3, 2, 1],      # cuenta
              [1, 6, 1, 2],       # devoluciones
              [2, 0, 17, 1],      # envio
              [1, 2, 0, 37]])     # facturacion
print(ETIQUETAS)

soporte = M.sum(axis=1)
precision = np.diag(M) / M.sum(axis=0)
recall = np.diag(M) / soporte
f1 = 2 * precision * recall / (precision + recall)
f1_macro = f1.mean()
f1_pond = (f1 * soporte).sum() / soporte.sum()

# Controllo automatico
assert all(v is not None for v in (precision, recall, f1, f1_macro, f1_pond)), "Completa le righe contrassegnate con TODO"
assert np.isclose(recall[1], 6 / 10), "il richiamo di devoluciones è 6/10 = 0,6"
y_real = [ETIQUETAS[i] for i in range(4) for j in range(4) for _ in range(M[i, j])]
y_pred = [ETIQUETAS[j] for i in range(4) for j in range(4) for _ in range(M[i, j])]
p, r, f, s = precision_recall_fscore_support(y_real, y_pred, labels=ETIQUETAS)
assert np.allclose(precision, p), "precisione = diagonale / somma della colonna"
assert np.allclose(recall, r), "richiamo = diagonale / somma della riga"
assert np.allclose(f1, f), "F1 = 2 · P · R / (P + R)"
assert np.isclose(f1_macro, f1_score(y_real, y_pred, average="macro")), "l'F1 macro è la media semplice degli F1"
assert np.isclose(f1_pond, f1_score(y_real, y_pred, average="weighted")), "il ponderato usa come pesi il supporto di ogni classe"
assert f1_macro < f1_pond, "qui il ponderato è maggiore: la classe piccola (devoluciones) è la peggiore"
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
def crear_pipeline_svm():
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", LinearSVC())])


def validar(df):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    return cross_val_score(crear_pipeline_svm(), df["texto"], df["categoria"], cv=skf)

# Controllo automatico
pipe = crear_pipeline_svm()
assert pipe is not None, "Completa le righe contrassegnate con TODO"
assert isinstance(pipe, Pipeline) and [n for n, _ in pipe.steps] == ["tfidf", "clf"], "la Pipeline deve avere i passi «tfidf» e «clf»"
assert isinstance(pipe.named_steps["tfidf"], TfidfVectorizer) and pipe.named_steps["tfidf"].ngram_range == (1, 2)
assert isinstance(pipe.named_steps["clf"], LinearSVC)
assert not hasattr(pipe.named_steps["tfidf"], "vocabulary_"), "la Pipeline si restituisce senza adattarla"
for idioma in ("spa", "ita"):
    df = tickets[idioma]
    puntos = validar(df)
    assert puntos is not None, "Completa le righe contrassegnate con TODO"
    assert len(puntos) == 5, "5 partizioni, 5 valori"
    manual = []
    for idx_ent, idx_prueba in StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(df["texto"], df["categoria"]):
        modelo = crear_pipeline_svm().fit(df["texto"].iloc[idx_ent], df["categoria"].iloc[idx_ent])
        manual.append(modelo.score(df["texto"].iloc[idx_prueba], df["categoria"].iloc[idx_prueba]))
    assert np.allclose(puntos, manual), "deve coincidere con il calcolo manuale: passa l'intera Pipeline a cross_val_score"
    assert puntos.mean() > 0.8, "con TF-IDF e LinearSVC ci si aspetta più di 0,8"
    print(idioma, "accuratezza media", round(puntos.mean(), 3), "±", round(puntos.std(), 3))
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
def submuestrear(df, cuotas, semilla=42):
    partes = [g.sample(n=cuotas[c], random_state=semilla) for c, g in df.groupby("categoria")]
    return pd.concat(partes).reset_index(drop=True)


def evaluar(des):
    X_ent, X_prueba, y_ent, y_prueba = train_test_split(des["texto"], des["categoria"], test_size=0.3, stratify=des["categoria"], random_state=42)
    resultados = []
    for peso in (None, "balanced"):
        pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", LogisticRegression(max_iter=1000, class_weight=peso))]).fit(X_ent, y_ent)
        resultados.append(pipe.predict(X_prueba))
    exactitud_sin = accuracy_score(y_prueba, resultados[0])
    f1_sin = f1_score(y_prueba, resultados[0], average="macro")
    f1_balanced = f1_score(y_prueba, resultados[1], average="macro")
    return exactitud_sin, f1_sin, f1_balanced

# Controllo automatico
for idioma in ("spa", "ita"):
    des = submuestrear(tickets[idioma], CUOTAS_P)
    assert des is not None, "Completa le righe contrassegnate con TODO"
    assert des["categoria"].value_counts().to_dict() == CUOTAS_P, "il DataFrame deve avere esattamente le quote richieste"
    assert list(des.index) == list(range(143)), "reimposta l'indice (reset_index(drop=True))"
    assert des.equals(submuestrear(tickets[idioma], CUOTAS_P)), "con lo stesso seme il risultato deve ripetersi"
    res = evaluar(des)
    assert all(v is not None for v in res), "Completa le righe contrassegnate con TODO"
    exactitud_sin, f1_sin, f1_bal = res
    assert exactitud_sin > f1_sin + 0.15, "senza pesi l'accuratezza dovrebbe essere molto maggiore dell'F1 macro"
    assert f1_bal > f1_sin + 0.1, "class_weight=\"balanced\" dovrebbe migliorare l'F1 macro"
    print(idioma, "accuratezza senza pesi", round(exactitud_sin, 3), "| F1 macro senza pesi", round(f1_sin, 3), "| F1 macro balanced", round(f1_bal, 3))
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
def top_palabras(pipe, k=8):
    palabras = pipe.named_steps["tfidf"].get_feature_names_out()
    coef = pipe.named_steps["clf"].coef_
    clases = pipe.named_steps["clf"].classes_
    return {clase: [palabras[j] for j in np.argsort(coef[i])[::-1][:k]] for i, clase in enumerate(clases)}


def n_veredictos(df):
    return sum(1 for t in df["texto"] if separar_veredicto(t)[1] != "")

# Controllo automatico
FACTURACION = {"spa": {"cobro", "factura", "pago", "importe", "tarjeta", "cargo", "cobrado", "pagos"},
               "ita": {"addebito", "fattura", "pagamento", "importo", "addebitato", "carta", "pagato", "pagare"}}
for idioma in ("spa", "ita"):
    df = tickets[idioma]
    pipe = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression(max_iter=1000))]).fit(df["texto"], df["categoria"])
    top = top_palabras(pipe, k=8)
    assert top is not None and n_veredictos(df) is not None, "Completa le righe contrassegnate con TODO"
    assert sorted(top) == ETIQUETAS and all(len(v) == 8 for v in top.values()), "una lista di 8 parole per classe"
    fila = list(pipe.named_steps["clf"].classes_).index("facturacion")
    coef = pipe.named_steps["clf"].coef_[fila]
    palabras = list(pipe.named_steps["tfidf"].get_feature_names_out())
    assert top["facturacion"][0] == palabras[int(np.argmax(coef))], "la prima parola deve essere quella con il coefficiente maggiore"
    coefs_top = [coef[palabras.index(p)] for p in top["facturacion"]]
    assert coefs_top == sorted(coefs_top, reverse=True), "ordina dal coefficiente maggiore al minore"
    assert len(set(top["facturacion"][:5]) & FACTURACION[idioma]) >= 3, "le parole di fatturazione dovrebbero dominare l'elenco"
    assert n_veredictos(df) == sum(1 for t in df["texto"] if separar_veredicto(t)[1]) and n_veredictos(df) > 100, "conta i ticket con frase finale corta"
    print(idioma, "facturacion:", top["facturacion"], "| ticket con verdetto:", n_veredictos(df))
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
def diferencia_veredicto(df):
    cuerpos = df["texto"].map(lambda t: separar_veredicto(t)[0])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", LogisticRegression(max_iter=1000))])
    con = cross_val_score(pipe, df["texto"], df["categoria"], cv=skf).mean()
    sin = cross_val_score(pipe, cuerpos, df["categoria"], cv=skf).mean()
    return con, sin

def _cv(textos, y):
    pipe = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2))), ("clf", LogisticRegression(max_iter=1000))])
    return cross_val_score(pipe, textos, y, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)).mean()


for idioma in ("spa", "ita"):
    df = tickets[idioma]
    resultado = diferencia_veredicto(df)
    assert resultado is not None, "Completa le righe contrassegnate con TODO"
    con, sin = resultado
    assert con is not None and sin is not None, "Completa le righe contrassegnate con TODO"
    esperado_sin = _cv(df["texto"].map(lambda t: separar_veredicto(t)[0]), df["categoria"])
    assert np.isclose(con, _cv(df["texto"], df["categoria"])), "valore con verdetto: validazione incrociata sui testi completi"
    assert np.isclose(sin, esperado_sin), "valore senza verdetto: validazione incrociata sui testi senza la frase finale corta"
    assert con - sin > 0.01, "togliere il verdetto dovrebbe abbassare l'accuratezza"
    print(idioma, "con verdetto", round(con, 4), "| senza verdetto", round(sin, 4))
print("Sfida finale corretta ✔")

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto della lezione non ti è rimasto chiaro (Naive Bayes e lisciamento, precisione contro richiamo, F1 macro contro ponderato, fuga di informazione, coefficienti, validazione incrociata)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*